# collaborative filtering v1

- direct collaboration scoring
- item-item collaborative filtering
- same holdout style as `design_v1`


In [1]:
from __future__ import annotations

from ast import literal_eval
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
DATA_DIR = ROOT / "data" / "synthetic"


## Load


In [2]:
collab_table = pd.read_csv(DATA_DIR / "collaboration_training_table.csv")
collabs = pd.read_csv(DATA_DIR / "collaborations.csv")

collab_table["lastEventAt"] = pd.to_datetime(collab_table["lastEventAt"], utc=True)
collab_table["tagsKey"] = collab_table["tagsKey"].apply(literal_eval)
collabs["tagsKey"] = collabs["tagsKey"].apply(literal_eval)

active_collabs = collabs[collabs["status"].isin(["submission", "voting"])].copy()
active_collabs = active_collabs[["id", "projectId", "name", "status", "tagsKey", "publishedAt"]].rename(columns={"id": "collaborationId"})

collab_table.head()

,userId,projectId,collaborationId,collaboration_favorite,collaboration_like,submission_favorite,submission_like,submission_vote,totalEventWeight,lastEventAt,status,tags,tagsKey,publishedAt
0,user-0001,project-007,collab-007-04,0,0,1,1,0,3.00,2026-04-24 00:22:43+00:00,submission,"['industrial', 'uk-bass']","[industrial, uk-bass]",2026-01-10T03:32:07Z
1,user-0001,project-003,collab-003-01,0,1,1,3,0,5.75,2026-04-03 07:12:35+00:00,submission,"['breakbeat', 'dub', 'jungle']","[breakbeat, dub, jungle]",2026-01-21T00:43:40Z
2,user-0001,project-009,collab-009-02,0,0,1,3,0,5.00,2026-03-28 11:54:51+00:00,voting,['acid'],[acid],2026-01-16T19:47:28Z
3,user-0002,project-007,collab-007-01,0,0,2,2,0,6.00,2026-04-16 05:06:49+00:00,completed,['ambient'],[ambient],2026-02-04T05:59:49Z
4,user-0002,project-012,collab-012-02,0,0,2,2,0,6.00,2026-04-04 14:53:01+00:00,submission,"['breakbeat', 'dub']","[breakbeat, dub]",2026-04-14T19:16:02Z


## Holdout split


In [3]:
eligible_history = collab_table[collab_table["status"].isin(["submission", "voting"])].copy()
eligible_history = eligible_history.sort_values(["userId", "lastEventAt"])

holdout = eligible_history.groupby("userId").tail(1).copy()
train_history = eligible_history.merge(
    holdout[["userId", "collaborationId"]].assign(_holdout=1),
    on=["userId", "collaborationId"],
    how="left",
)
train_history = train_history[train_history["_holdout"].isna()].drop(columns="_holdout")

valid_users = train_history["userId"].value_counts()
valid_users = set(valid_users[valid_users > 0].index) & set(holdout["userId"])
train_history = train_history[train_history["userId"].isin(valid_users)].copy()
holdout = holdout[holdout["userId"].isin(valid_users)].copy()

len(valid_users), train_history.shape, holdout.shape

(239, (1434, 14), (239, 14))

## User-collaboration matrix


In [4]:
interaction_df = (
    train_history.groupby(["userId", "collaborationId"], as_index=False)["totalEventWeight"]
    .sum()
)

interaction_matrix = interaction_df.pivot(index="userId", columns="collaborationId", values="totalEventWeight").fillna(0.0)
interaction_matrix.shape

(239, 38)

## Similarity


In [5]:
X = interaction_matrix.to_numpy(dtype=float)
item_matrix = X.T
norms = np.linalg.norm(item_matrix, axis=1, keepdims=True)
norms[norms == 0] = 1.0
item_matrix_norm = item_matrix / norms
similarity = item_matrix_norm @ item_matrix_norm.T
np.fill_diagonal(similarity, 0.0)

item_ids = interaction_matrix.columns.tolist()
sim_df = pd.DataFrame(similarity, index=item_ids, columns=item_ids)
sim_df.iloc[:5, :5]

,collab-001-01,collab-001-02,collab-002-01,collab-002-02,collab-002-03
collab-001-01,0.000000,0.141247,0.241948,0.004985,0.077898
collab-001-02,0.141247,0.000000,0.104791,0.260399,0.084873
collab-002-01,0.241948,0.104791,0.000000,0.000000,0.101494
collab-002-02,0.004985,0.260399,0.000000,0.000000,0.030545
collab-002-03,0.077898,0.084873,0.101494,0.030545,0.000000


## Score candidates


In [6]:
popularity = interaction_df.groupby("collaborationId", as_index=False)["totalEventWeight"].sum()
popularity["popularityNorm"] = popularity["totalEventWeight"] / (popularity["totalEventWeight"].max() or 1)
pop_map = dict(zip(popularity["collaborationId"], popularity["popularityNorm"]))

active_ids = active_collabs["collaborationId"].tolist()
candidate_rows = []

for user_id, row in interaction_matrix.iterrows():
    seen_weights = row[row > 0]
    seen_ids = set(seen_weights.index)
    user_scores = {}

    if len(seen_weights) > 0:
        seen_vector = seen_weights.to_numpy(dtype=float)
        sims = sim_df[seen_weights.index].mul(seen_vector, axis=1).sum(axis=1)
        user_scores = sims.to_dict()

    for collab_id in active_ids:
        if collab_id in seen_ids:
            continue
        cf_score = float(user_scores.get(collab_id, 0.0))
        popularity_score = float(pop_map.get(collab_id, 0.0))
        final_score = 0.85 * cf_score + 0.15 * popularity_score
        candidate_rows.append({
            "userId": user_id,
            "collaborationId": collab_id,
            "cfScore": cf_score,
            "popularityNorm": popularity_score,
            "v1Score": final_score,
        })

candidate_scores = pd.DataFrame(candidate_rows)
candidate_scores = candidate_scores.merge(active_collabs, on="collaborationId", how="left")
candidate_scores.head()

,userId,collaborationId,cfScore,popularityNorm,v1Score,projectId,name,status,tagsKey,publishedAt
0,user-0001,collab-001-01,1.462580,0.716581,1.350680,project-001,Collaboration 1-1,voting,"[acid, breakbeat, cinematic]",2026-04-28T14:26:30Z
1,user-0001,collab-001-02,0.994962,0.894814,0.979940,project-001,Collaboration 1-2,voting,"[breakbeat, lofi, organic]",2025-12-17T02:06:25Z
2,user-0001,collab-002-01,1.124193,0.341125,1.006733,project-002,Collaboration 2-1,submission,"[ambient, drum-and-bass]",2026-03-23T22:33:55Z
3,user-0001,collab-002-02,0.637364,0.327977,0.590956,project-002,Collaboration 2-2,voting,"[idm, uk-bass]",2026-01-16T13:18:16Z
4,user-0001,collab-002-03,1.516615,0.322133,1.337442,project-002,Collaboration 2-3,submission,[minimal],2026-03-20T00:06:58Z


## Top recommendations


In [7]:
top_recommendations = (
    candidate_scores
    .sort_values(["userId", "v1Score"], ascending=[True, False])
    .groupby("userId")
    .head(10)
    .copy()
)

top_recommendations["rank"] = top_recommendations.groupby("userId").cumcount() + 1
top_recommendations = top_recommendations[[
    "userId",
    "rank",
    "projectId",
    "collaborationId",
    "name",
    "status",
    "tagsKey",
    "cfScore",
    "popularityNorm",
    "v1Score",
]]
top_recommendations.head(20)

,userId,rank,projectId,collaborationId,name,status,tagsKey,cfScore,popularityNorm,v1Score
24,user-0001,1,project-010,collab-010-03,Collaboration 10-3,voting,[acid],3.743715,0.637692,3.277811
30,user-0001,2,project-012,collab-012-02,Collaboration 12-2,submission,"[breakbeat, dub]",2.607491,0.412710,2.278274
26,user-0001,3,project-011,collab-011-02,Collaboration 11-2,voting,"[ambient, lofi]",2.461804,0.691746,2.196295
31,user-0001,4,project-012,collab-012-04,Collaboration 12-4,submission,"[acid, breakbeat, trance]",2.387247,0.804967,2.149905
19,user-0001,5,project-008,collab-008-02,Collaboration 8-2,submission,"[ambient, electro]",2.395429,0.362308,2.090461
16,user-0001,6,project-007,collab-007-03,Collaboration 7-3,submission,"[organic, trance]",2.246320,0.587290,1.997465
10,user-0001,7,project-005,collab-005-01,Collaboration 5-1,submission,"[breakbeat, synthwave]",2.148983,0.513514,1.903663
6,user-0001,8,project-003,collab-003-03,Collaboration 3-3,submission,"[cinematic, lofi, minimal]",2.060178,0.769175,1.866527
27,user-0001,9,project-011,collab-011-03,Collaboration 11-3,submission,"[dub, minimal, synthwave]",1.996894,0.587290,1.785453
7,user-0001,10,project-004,collab-004-01,Collaboration 4-1,voting,"[ambient, lofi, minimal]",1.908039,0.803506,1.742359


## Sanity evaluation


In [8]:
history_tags = train_history.groupby("userId")["tagsKey"].apply(lambda rows: set(t for arr in rows for t in arr)).to_dict()
history_projects = train_history.groupby("userId")["projectId"].apply(lambda s: set(s)).to_dict()

sanity_eval = top_recommendations.copy()
sanity_eval["tagOverlap"] = [len(set(tags) & history_tags.get(uid, set())) for uid, tags in zip(sanity_eval["userId"], sanity_eval["tagsKey"])]
sanity_eval["sameSeenProject"] = [pid in history_projects.get(uid, set()) for uid, pid in zip(sanity_eval["userId"], sanity_eval["projectId"])]

sanity_summary = {
    "rows": int(len(sanity_eval)),
    "users": int(sanity_eval["userId"].nunique()),
    "unique_recommended_collabs": int(sanity_eval["collaborationId"].nunique()),
    "catalog_coverage_pct": round(100 * sanity_eval["collaborationId"].nunique() / len(collabs), 2),
    "active_only": bool(sanity_eval["status"].isin(["submission", "voting"]).all()),
    "top10_tag_overlap_pct": round(100 * (sanity_eval["tagOverlap"] > 0).mean(), 2),
    "top10_same_project_seen_pct": round(100 * sanity_eval["sameSeenProject"].mean(), 2),
    "top1_tag_overlap_pct": round(100 * (sanity_eval[sanity_eval["rank"] == 1]["tagOverlap"] > 0).mean(), 2),
    "top1_same_project_seen_pct": round(100 * sanity_eval[sanity_eval["rank"] == 1]["sameSeenProject"].mean(), 2),
}

sanity_summary

{'rows': 2390,
 'users': 239,
 'unique_recommended_collabs': 38,
 'catalog_coverage_pct': 80.85,
 'active_only': True,
 'top10_tag_overlap_pct': np.float64(74.31),
 'top10_same_project_seen_pct': np.float64(29.79),
 'top1_tag_overlap_pct': np.float64(79.92),
 'top1_same_project_seen_pct': np.float64(25.1)}

In [9]:
top_recommendations[top_recommendations["rank"] == 1]["collaborationId"].value_counts().head(10)

collaborationId
collab-012-04    50
collab-004-01    39
collab-009-03    24
collab-010-03    18
collab-011-04    13
collab-006-02    12
collab-003-01    11
collab-004-02     9
collab-001-02     9
collab-011-03     6
Name: count, dtype: int64

## Holdout evaluation


In [10]:
holdout_eval = holdout[["userId", "collaborationId"]].rename(columns={"collaborationId": "heldOutCollaborationId"})
hits = top_recommendations.merge(holdout_eval, on="userId", how="inner")
hits["isHit"] = hits["collaborationId"] == hits["heldOutCollaborationId"]

user_hits = hits.groupby("userId").agg(
    hitAt10=("isHit", "max"),
    reciprocalRank=("rank", lambda s: 1 / s[hits.loc[s.index, "isHit"]].iloc[0] if hits.loc[s.index, "isHit"].any() else 0),
).reset_index()

holdout_summary = {
    "users_evaluated": int(len(user_hits)),
    "hit_rate_at_10": round(float(user_hits["hitAt10"].mean()), 4),
    "mrr_at_10": round(float(user_hits["reciprocalRank"].mean()), 4),
}

holdout_summary

{'users_evaluated': 239, 'hit_rate_at_10': 0.3096, 'mrr_at_10': 0.0853}